In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("piyshsss/dentex")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'dentex' dataset.
Path to dataset files: /kaggle/input/dentex


In [2]:
!pip install -q ultralytics
import os
import json
import shutil
!pip install -q ultralytics
from tqdm import tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
libraft-cu12 26.2.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
libcuvs-cu12 26.2.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
cuda-python 12.9.7 requires cuda-bindings~=12.9.7, but you have cuda-bindings 13.3.1 which is incompatible.


In [3]:
import json

json_path = "/kaggle/input/dentex/DENTEX/training_d/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json"

with open(json_path) as f:
    data = json.load(f)

print(data.keys())

dict_keys(['images', 'annotations', 'categories_1', 'categories_2', 'categories_3'])


In [4]:
print(data["annotations"][0])

{'iscrowd': 0, 'image_id': 1, 'bbox': [542.0, 698.0, 220.0, 271.0], 'segmentation': [[621, 703, 573, 744, 542, 885, 580, 945, 650, 969, 711, 883, 762, 807, 748, 741, 649, 698]], 'id': 1, 'area': 39683, 'category_id_1': 3, 'category_id_2': 7, 'category_id_3': 0}


In [5]:
import os

os.makedirs("/kaggle/working/images/train",exist_ok=True)
os.makedirs("/kaggle/working/labels/train",exist_ok=True)

In [6]:
import shutil

src = "/kaggle/input/dentex/DENTEX/training_d/quadrant-enumeration-disease/xrays"

for file in os.listdir(src):
    shutil.copy(
        os.path.join(src,file),
        "/kaggle/working/images/train"
    )

In [7]:
import os

images = {
    img["id"]:img
    for img in data["images"]
}

for ann in data["annotations"]:

    img_info = images[ann["image_id"]]

    w = img_info["width"]
    h = img_info["height"]

    txt_name = img_info["file_name"].replace(".png",".txt")

    label_path = f"/kaggle/working/labels/train/{txt_name}"

    polygon = ann["segmentation"][0]

    coords = []

    for i in range(0,len(polygon),2):

        x = polygon[i]/w
        y = polygon[i+1]/h

        coords.extend([x,y])

    class_id = 0

    line = str(class_id)

    for c in coords:
        line += f" {c:.6f}"

    with open(label_path,"a") as f:
        f.write(line+"\n")

In [8]:
import json

json_path = "/kaggle/input/dentex/DENTEX/validation_data/quadrant_enumeration_disease/validation.json"

with open(json_path) as f:
    data = json.load(f)

print(data.keys())

dict_keys(['images', 'annotations', 'categories_1', 'categories_2', 'categories_3'])


In [9]:
import os

os.makedirs("/kaggle/working/images/val",exist_ok=True)
os.makedirs("/kaggle/working/labels/val",exist_ok=True)

In [10]:
import shutil

src = "/kaggle/input/dentex/DENTEX/validation_data/quadrant_enumeration_disease/xrays"

for file in os.listdir(src):
    shutil.copy(
        os.path.join(src,file),
        "/kaggle/working/images/val"
    )

In [11]:
import os

images = {
    img["id"]:img
    for img in data["images"]
}

for ann in data["annotations"]:

    img_info = images[ann["image_id"]]

    w = img_info["width"]
    h = img_info["height"]

    txt_name = img_info["file_name"].replace(".png",".txt")

    label_path = f"/kaggle/working/labels/val/{txt_name}"

    polygon = ann["segmentation"][0]

    coords = []

    for i in range(0,len(polygon),2):

        x = polygon[i]/w
        y = polygon[i+1]/h

        coords.extend([x,y])

    class_id = 0

    line = str(class_id)

    for c in coords:
        line += f" {c:.6f}"

    with open(label_path,"a") as f:
        f.write(line+"\n")

In [12]:
%%writefile /kaggle/working/data.yaml

path: /kaggle/working

train: images/train
val: images/val

names:
  0: tooth

Overwriting /kaggle/working/data.yaml


In [13]:
from ultralytics import YOLO

model = YOLO("yolov8m-seg.pt")

In [14]:
%pip uninstall -y torch torchvision torchaudio
%pip cache purge
%pip install --upgrade pip setuptools wheel
%pip install --no-cache-dir torch torchvision torchaudio
%pip install --no-cache-dir --upgrade ultralytics

Found existing installation: torch 2.13.0
Uninstalling torch-2.13.0:
  Successfully uninstalled torch-2.13.0
Found existing installation: torchvision 0.28.0
Uninstalling torchvision-0.28.0:
  Successfully uninstalled torchvision-0.28.0
Files removed: 134 (2733.9 MB)
Directories removed: 0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 225.7 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 151.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 169.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchvision]


In [15]:


results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=50,
    imgsz=1024,
    batch=8,
    device=0,
    workers=4
)
metrics = model.val()

Ultralytics 8.4.91 🚀 Python-3.12.13 torch-2.13.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

KeyboardInterrupt: 